# 🏢 مختبر مسكن ثلاثي الأبعاد: مسار التدريب عبر Kaggle (GPU T4)
### Maskan3D Lab: 3D Gaussian Splatting Training Pipeline on Kaggle

تم تخصيص هذا الدفتر لبيئة **Kaggle** (التي تمنحك 30 ساعة أسبوعياً بذاكرة رام 30GB وبطاقة GPU T4).

---
### ⚠️ إعدادات هامة جداً قبل التشغيل في Kaggle:
من الشريط الجانبي الأيمن في Kaggle (Notebook options):
1. **Accelerator:** اختر **GPU T4 x2** (أو GPU P100).
2. **Internet:** فعّل خيار **Internet ON** (ضروري جداً حتى يتمكن الدفتر من تحميل الحزم).


## ⚙️ إعداد المتغيرات والمسارات الرئيسية (Kaggle Working Paths)
المسارات هنا مهيأة للعمل داخل مسار `/kaggle/working` المخصص للكتابة والتصدير في كاجل.

In [ ]:
# ========================================================
# إعدادات المسارات والمتغيرات الرئيسية في Kaggle
# ========================================================
import os
from pathlib import Path

# مجلد العمل الرئيسي في كاجل
WORKING_DIR = Path("/kaggle/working")

# مسار ملف الصور المضغوط (قم برفعه في /kaggle/working أو سحبه للملفات)
ZIP_PHOTOS_PATH = WORKING_DIR / "photos.zip"

# مجلد فك ضغط الصور
IMAGES_DIR = WORKING_DIR / "data/images"

# مجلد حفظ مخرجات البيانات المعالجة عبر COLMAP
PROCESSED_DATA_DIR = WORKING_DIR / "data/nerfstudio/apartment"

# مجلد حفظ مخرجات ونقاط تفتيش التدريب
TRAIN_OUTPUTS_DIR = WORKING_DIR / "outputs"

# مجلد تصدير الملفات النهائية
EXPORTS_DIR = WORKING_DIR / "exports"

# اسم التجربة وعدد خطوات التدريب
EXPERIMENT_NAME = "apartment_splat"
MAX_ITERATIONS = 5000

print(f"📦 Photos Zip Path:       {ZIP_PHOTOS_PATH}")
print(f"📸 Images Target Dir:      {IMAGES_DIR}")
print(f"💾 Train Outputs Dir:      {TRAIN_OUTPUTS_DIR}")
print(f"🎯 Max Iterations:         {MAX_ITERATIONS}")


## 1️⃣ فحص بطاقة الرسوميات (Check GPU & Environment)
التحقق من توفر بطاقة الـ GPU في كاجل ومكتبات CUDA.

In [ ]:
# التحقق من مواصفات بطاقة الرسوميات
!nvidia-smi

# التحقق من إصدار بايثون و PyTorch
import sys
import torch

print(f"🐍 Python Version: {sys.version.split()[0]}")
print(f"🔥 PyTorch Version: {torch.__version__}")
print(f"⚡ CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"🎮 GPU Device Name: {torch.cuda.get_device_name(0)}")
    total_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"🧠 Total VRAM: {total_mem_gb:.2f} GB")
else:
    print("⚠️ تحذير: لم يتم تفعيل الـ GPU! تأكد من اختيار Accelerator: GPU T4 x2 من القائمة اليمنى.")


## 2️⃣ تثبيت الحزم والمتطلبات (Install Dependencies)
- تثبيت **COLMAP** و **FFmpeg** عبر `apt-get`.
- تثبيت **nerfstudio** عبر `pip`.

In [ ]:
# تحديث الحزم وتثبيت colmap و ffmpeg
!apt-get update -y && apt-get install -y colmap ffmpeg

# ترقية مدير الحزم pip وتثبيت nerfstudio
!pip install --upgrade pip
!pip install nerfstudio


## 3️⃣ رفع وفك ضغط صور الشقة (Unzip Photos Flat)
> **طريقة الرفع في Kaggle:**
> 1. اضغط على أيقونة الرفع **Upload** (في أعلى الشريط الأيمن أو شريط الأدوات) وارفع ملف `photos.zip`، أو ضعه داخل `/kaggle/working/photos.zip`.
> 2. ستقوم الخلية بفك ضغط جميع الصور بشكل مسطح داخل مجلد الصور.

In [ ]:
import os
import shutil
import zipfile
from pathlib import Path

target_dir = Path(IMAGES_DIR)
target_dir.mkdir(parents=True, exist_ok=True)
valid_extensions = {'.jpg', '.jpeg', '.png', '.webp'}

print('🔍 جاري فحص ملفات الصور في Kaggle...')

# 1. فحص إذا كانت الصور مفكوكة تلقائياً داخل /kaggle/input (سلوك كاجل الافتراضي عند رفع Dataset)
input_images = []
input_path = Path('/kaggle/input')
if input_path.exists():
    for p in input_path.rglob('*'):
        if p.is_file() and p.suffix.lower() in valid_extensions:
            if not p.name.startswith('.') and '__MACOSX' not in str(p):
                input_images.append(p)

# 2. فحص إذا كان هناك ملف أرشيف zip في /kaggle/input أو /kaggle/working
zip_files = []
if input_path.exists():
    zip_files.extend(list(input_path.rglob('*.zip')))
working_path = Path('/kaggle/working')
if working_path.exists():
    zip_files.extend(list(working_path.glob('*.zip')))

copied_count = 0

if input_images:
    print(f'📸 تم العثور على {len(input_images)} صورة داخل مجلد المدخلات (/kaggle/input/)')
    for img in input_images:
        dest = target_dir / img.name
        shutil.copy2(img, dest)
        copied_count += 1
    print(f'✅ تم نسخ الصور بنجاح إلى مجلد المعالجة: {target_dir}')

elif zip_files:
    zip_to_use = zip_files[0]
    print(f'📦 تم العثور على أرشيف مضغوط: {zip_to_use}')
    with zipfile.ZipFile(zip_to_use, 'r') as zf:
        for member in zf.infolist():
            if member.is_dir():
                continue
            filename = os.path.basename(member.filename)
            if not filename or filename.startswith('.') or '__MACOSX' in member.filename:
                continue
            ext = os.path.splitext(filename)[1].lower()
            if ext in valid_extensions:
                dest_path = target_dir / filename
                with zf.open(member) as source, open(dest_path, 'wb') as target:
                    target.write(source.read())
                copied_count += 1
    print(f'✅ تم فك الضغط بنجاح إلى: {target_dir}')

else:
    # فحص إذا كانت الصور موجودة بالفعل في المجلد المستهدف
    existing = [p for p in target_dir.iterdir() if p.is_file() and p.suffix.lower() in valid_extensions]
    copied_count = len(existing)
    if copied_count > 0:
        print(f'✅ الصور موجودة بالفعل في مجلد العمل: {copied_count} صورة.')
    else:
        print('❌ لم يتم العثور على أي صور أو ملف مضغوط في /kaggle/input أو /kaggle/working!')

print('=' * 50)
print(f'📸 إجمالي الصور الجاهزة للمعالجة: {copied_count}')
print(f'📁 مسار مجلد الصور: {target_dir}')
print('=' * 50)

## 4️⃣ استخراج مواضع الكاميرات من الصور عبر COLMAP (Process Images)
تشغيل أمر `ns-process-data images` مع خيار `--no-gpu`.

In [ ]:
# تنظيف المجلد السابق وتحديد المسارات بأمان
import shutil
from pathlib import Path

# المسارات الافتراضية في كاجل لضمان عدم ظهور NameError إذا لم يتم تشغيل الخلية الأولى
WORKING_DIR = Path("/kaggle/working")
IMAGES_DIR = IMAGES_DIR if "IMAGES_DIR" in globals() else WORKING_DIR / "data/images"
PROCESSED_DATA_DIR = PROCESSED_DATA_DIR if "PROCESSED_DATA_DIR" in globals() else WORKING_DIR / "data/nerfstudio/apartment"

if Path(PROCESSED_DATA_DIR).exists():
    shutil.rmtree(PROCESSED_DATA_DIR)

# تشغيل المعالجة بالمطابقة الشاملة
!ns-process-data images \
    --data "{IMAGES_DIR}" \
    --output-dir "{PROCESSED_DATA_DIR}" \
    --matching-method exhaustive \
    --no-gpu

## 📊 فحص نسبة تسجيل الكاميرات (Registration Rate Evaluation)
قراءة ملف `transforms.json` الناتج عن COLMAP لحساب نسبة الصور المسجلة مقارنة بإجمالي الصور المرفوعة.

In [ ]:
import json
from pathlib import Path

transforms_file = Path(PROCESSED_DATA_DIR) / "transforms.json"

if not transforms_file.exists():
    print(f"❌ خطأ: لم يتم العثور على {transforms_file}!")
else:
    with open(transforms_file, "r", encoding="utf-8") as f:
        data = json.load(f)
    
    registered_count = len(data.get("frames", []))
    
    source_dir = Path(IMAGES_DIR)
    exts = ("*.jpg", "*.jpeg", "*.png", "*.webp", "*.JPG", "*.JPEG", "*.PNG")
    uploaded_files = []
    for ext in exts:
        uploaded_files.extend(list(source_dir.glob(ext)))
    uploaded_count = len(uploaded_files)
    
    if uploaded_count == 0:
        uploaded_count = registered_count

    ratio = (registered_count / uploaded_count * 100) if uploaded_count > 0 else 0.0

    print("=" * 60)
    print(f"📸 إجمالي الصور المرفوعة (Uploaded Images):    {uploaded_count}")
    print(f"✅ الصور المسجلة بنجاح (Registered Cameras):   {registered_count}")
    print(f"📈 نسبة التسجيل (Registration Percentage):      {ratio:.1f}%")
    print("=" * 60)

    if ratio < 70.0:
        print(f"⚠️ تحذير: نسبة التسجيل منخفضة ({ratio:.1f}% < 70%).")
    else:
        print("🎉 ممتاز! نسبة التسجيل تتجاوز 70%، المشهد جاهز للتدريب.")


## 5️⃣ تدريب نموذج Splatfacto (Train 3D Gaussian Splatting)
بدء تدريب النموذج باستخدام `ns-train splatfacto`.
- نحدد `MAX_JOBS=4` لأن كاجل يوفر رامات ضخمة (30GB) مما يسرع بناء كود CUDA بدون أي خطر لنفاد الذاكرة.
- تم ضبط عدد الخطوات بـ 5000 خطوة كافية جداً وسريعة.

In [ ]:
import os

# في كاجل الرامات 30 جيجابايت لذا يمكن استخدام 4 مسارات تجميع بأمان وسرعة فائقة
os.environ["MAX_JOBS"] = "4"

print(f"🚀 بدء تدريب Splatfacto لعدد {MAX_ITERATIONS} خطوة...")
print("⏳ يُرجى الانتظار حتى تنتهي الخلية وتظهر علامة الصح الخضراء...")

!MAX_JOBS=4 ns-train splatfacto \
    --data "{PROCESSED_DATA_DIR}" \
    --output-dir "{TRAIN_OUTPUTS_DIR}" \
    --experiment-name "{EXPERIMENT_NAME}" \
    --max-num-iterations {MAX_ITERATIONS} \
    --steps-per-save 1000 \
    --viewer.quit-on-train-completion True


## 6️⃣ تصدير المشهد وتنزيل ملف .ply (Export & Download)
نقوم بتصدير المشهد بصيغة `.ply`، وحساب حجمه بالميجابايت، وتوليد رابط تنزيل مباشر في المتصفح بالإضافة لحفظه في مخرجات Kaggle Output.

In [ ]:
import glob
import os
from pathlib import Path
from IPython.display import FileLink, display

# إصلاح التوافق مع PyTorch 2.6+ (weights_only=False)
try:
    import nerfstudio.utils.eval_utils as eval_utils
    eval_file = Path(eval_utils.__file__)
    content = eval_file.read_text(encoding="utf-8")
    if 'weights_only=False' not in content:
        content = content.replace(
            'torch.load(load_path, map_location="cpu")',
            'torch.load(load_path, map_location="cpu", weights_only=False)'
        )
        eval_file.write_text(content, encoding="utf-8")
        print("✅ تم تفعيل التوافق مع PyTorch 2.6+ بنجاح!")
except Exception as e:
    print(f"تنبيه التوافق: {e}")

# البحث عن ملف config.yml
config_files = list(Path(TRAIN_OUTPUTS_DIR).glob(f"**/{EXPERIMENT_NAME}/**/config.yml"))
if not config_files:
    config_files = list(Path(TRAIN_OUTPUTS_DIR).glob("**/config.yml"))

if not config_files:
    raise FileNotFoundError(f"❌ لم يتم العثور على أي ملف config.yml في: {TRAIN_OUTPUTS_DIR}.")

latest_config = max(config_files, key=os.path.getmtime)
print(f"📄 Config file found: {latest_config}")

local_export_dir = Path(EXPORTS_DIR)
local_export_dir.mkdir(parents=True, exist_ok=True)
ply_filename = f"{EXPERIMENT_NAME}.ply"

# تنفيذ أمر التصدير
!ns-export gaussian-splat \
    --load-config "{latest_config}" \
    --output-dir "{local_export_dir}" \
    --output-filename "{ply_filename}"

exported_file = local_export_dir / ply_filename

if exported_file.exists():
    file_size_bytes = exported_file.stat().st_size
    file_size_mb = file_size_bytes / (1024 * 1024)
    
    print("=" * 60)
    print("🎉 تم تصدير ملف الـ Gaussian Splat بنجاح!")
    print(f"📁 اسم الملف:              {ply_filename}")
    print(f"⚖️ حجم الملف (File Size):    {file_size_mb:.2f} MB")
    print(f"💾 المسار النهائي:          {exported_file}")
    print("=" * 60)
    print("⬇️ اضغط على الرابط أدناه لتنزيل الملف إلى جهازك مباشرة:")
    display(FileLink(str(exported_file)))
    print("\n(يمكنك أيضاً تنزيله من القائمة الجانبية في Kaggle تحت تبويب Output)")
else:
    print("❌ تعذر العثور على ملف .ply المصدر، يرجى مراجعة سجلات الخطأ أعلاه.")


## 🎯 الخطوات اللاحقة بعد التنزيل
1. قم بوضع ملف `apartment_splat.ply` في مجلد المشروع المحلي `maskan3d-lab/outputs/`.
2. افتح موقع **PlayCanvas SuperSplat** لمعاينة الشقة ثلاثية الأبعاد وتنظيف الشوائب.
